In [2]:
import os #Environment variables
from gpt4all import GPT4All
from llama_cpp import Llama
from jinja2 import Template
from jinja2 import Environment, BaseLoader
from datetime import datetime

#### Chat Templates
Chat templates are jinja templates bundled with model files (.gguf) used for rendering prompts before sending them to the model. They are rendered by filling placeholders with the prompt itself, roles (such as "system" for the prompt prepended at the beginning of the chat, "user" for the user query and "assistant" for the model response) and other relevant variables (such as the timestamp). 
- Though the templates are jinja files, it doesn't mean that the models read in python natively (it is just for setting the chat format in a human-readable way).
- On hugging face, for some models you can access the template from a tokenizer .json file available in the model specs (usually a page listing all the files that come bundled with the model). This will be the case for "open" models that allow public access to their chat template.
- Some "gated" models (such as Llama and Mistral) do not allow public access to their chat templates. For these cases it is recommended to read the template bundled with the local model file (.gguf)

**Sources**:

   - https://docs.gpt4all.io/gpt4all_desktop/chat_templates.html#how-do-i-find-a-chat-template

#### GPT4ALL & Chat Templates
- The GPT4All backend reads the chat template under the hood directly from the local .gguf model file (llama.cpp). 
- Though in python the chat template is usually stored as an attribute in the model instance, it will not be always the case for newer models.
- Thus for consistency, it is recommended to read the template directly from the local model file (.gguf) using the llama.cpp python api.

In [3]:
# Load Model pointing using GPU
model = GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf", device="cuda") 

In [ ]:
#---------------LOAD TEMPLATE FROM GPT4ALL MODEL CONFIG----------------
#----------MIGHT NOT ALWAYS BE AVAILABLE!!!!!!!!!!!!
#template=model.config["chatTemplate"]

In [4]:
#---------------------LOAD TEMPLATE FROM LOCAL GGUF MODEL FILE---------------------
#------RECOMMENDED: NOT DEPEND ON AVAILABILITY IN GPT4ALL API----------------

path=model.config['path'] #---------------->recommended: get gguf path from gpt4all model config for consistency

#1. Carga tu archivo GGUF local de forma normal
# (Usamos verbose=False para que no llene la pantalla con registros de inicialización)
llm = Llama(model_path=path, verbose=False,vocab_only=True)

# 2. Accede al diccionario de metadatos internos del GGUF
metadata = llm.metadata
template=metadata.get('tokenizer.chat_template')

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


In [5]:
#-------------RECOMMENDED FOR NEWER GPT4ALL MODEL TEMPLATES------------
#-----INJECT TIMESTAMP-----(REQUIRED BY GPT4ALL API)----------------
# 1. Definimos una función en Python que replique strftime_now
def strftime_now(formato):
    # Retorna la fecha de hoy con el formato que pida el template (ej: '%d %b %Y')
    return datetime.now().strftime(formato)

# 2. Creamos un entorno de Jinja e inyectamos la función como global
env = Environment(loader=BaseLoader())
env.globals.update(strftime_now=strftime_now)

# 3. Compilamos el texto de la plantilla usando este entorno seguro
# (Reemplaza 'raw_template_text' por la variable que extraes de model.config o GGUF)
jinja_template = env.from_string(template)

In [ ]:
#---------------IF LOADING TEMPLATE FROM LOCAL .GGUF, THIS CAN BE USED--------------------
#-----community .gguf adapted for llama.cpp usually contain simplified jinja templates, with hugging face/python datetime formatters removed
#-------this templates usually only require "user" and "assitant" code blocks
#jinja_template = Template(template)

In [7]:
#----------RENDER PROMPT---------------------

#-------------fake chat history for testing-----------------------
historial_chat = [
    {"role": "system", "content": "You are a helpful assistant."}
]
historial_chat.append({"role": "user", "content": "hola"})

# Renderizar el historial de forma idéntica a como lo pide el creador del modelo
    # Le inyectamos las variables que esperan las plantillas estándar de Hugging Face y GGUF
prompt_renderizado = jinja_template.render(
        messages=historial_chat,
        add_generation_prompt=True,
        bos_token="<s>",  # Tokens especiales por defecto si la plantilla los pide
        eos_token="</s>"
    )

In [8]:
print('---------chat template \n{}:'.format(template))

---------chat template 
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: 

In [9]:
print('-----------rendered template \n{}:'.format(prompt_renderizado))

-----------rendered template 
<s><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 12 Oct 2026

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

hola<|eot_id|><|start_header_id|>assistant<|end_header_id|>

:


In [11]:
model.close()
llm.close()